# 01: Measuring the net reduction of the river's flow on New Mexico's Middle Rio Grande

## Project Goal

The objective: to develop tools to analyze in detail the way the coupled human-natural system along New Mexico's Middle Rio Grande from Cochiti to San Marcial consumes water, and how patterns of consumption have changed over time.

The basic premise of this approach is something we call Net Flow Reduction (**NFR**): the difference between the amount of water that flows into the valley at Otowi and how much flows out at San Marcial. The simple mass balance difference is not a direct measure of consumptive use of water because of the complex interplay between inflows and outflows along the way. But it gives us a starting point to identify in greater detail the various factors - human, natural ecosystem, weather, and climate - that influence changes in net flow over time.

This has the potential to provide a valuable tool to inform decisions about the Middle Rio Grande's water use under the terms of the Rio Grande Compact, which creates a legal obligation to downstream users in southern New Mexico, Texas, and Mexico.


Analysis design and interpretation by John Fleck; code drafted with assistance from OpenAI Codex.


### Building a Reproducible Public Record

The analysis that follows uses the R programming language and the USGS *dataRetrieval* package to download daily streamflow values from the modern USGS Water Data API for the gages of interest for their entire available periods of record. The streamflow values are returned as a rate - cubic feet per second. The code converts them to a volume - acre-feet flowing past the gage of interest each day.

Because the returned data is not always complete, the code applies a completeness screen to enable later decisions about the quality of the data to be used for subsequent analysis and write a table of the resulting data to be used for later steps in the analytical process.

Required R packages: `dataRetrieval`, `dplyr`, `tidyr`, `readr`, `ggplot2`, and `scales`.


In [ ]:
library(dataRetrieval)
library(dplyr)
library(tidyr)
library(readr)
library(ggplot2)
library(scales)

# Find the public package root from either the notebook folder or the repository root.
find_project_root <- function(start = getwd()) {
  current <- normalizePath(start, mustWork = TRUE)

  repeat {
    if (dir.exists(file.path(current, "data", "nfr"))) {
      return(current)
    }

    parent <- dirname(current)
    if (identical(parent, current)) {
      stop("Could not find public package root containing 'data/nfr'.", call. = FALSE)
    }
    current <- parent
  }
}

project_root <- find_project_root()
output_path <- file.path(project_root, "data", "nfr", "net_flow_reduction_annual_flows_af.csv")

# USGS parameter 00060 is discharge in cubic feet per second.
# Statistic 00003 identifies the daily mean value.
parameter_code <- "00060"
statistic_id <- "00003"
query_start <- "1900-01-01"
final_analysis_year <- 2025
query_end <- paste0(final_analysis_year, "-12-31")
completeness_threshold <- 95
acre_feet_per_cfs_day <- 86400 / 43560

cat("Project root:", basename(project_root), "\n")
cat("Output table:", file.path("data", "nfr", basename(output_path)), "\n")


## The Gages

The analysis uses three gages - one at the upstream end of the Middle Rio Grande Valley and two at the downstream end.

- `08313000`: inflow Rio Grande at Otowi Bridge
- `08358400`: outflow through the Rio Grande Floodway at San Marcial
- `08358300`: outflow through the Low Flow Conveyance Channel at San Marcial, adjacent to the main river channel

Total outflow is calculated by summing the Rio Grande Floodway and LFCC.


In [ ]:
# Define the gages and the role each one plays in the accounting measure.
sites <- data.frame(
  site_no = c("08313000", "08358400", "08358300"),
  short_name = c("Otowi", "San Marcial Floodway", "San Marcial Conveyance"),
  role = c("inflow", "outflow_main_channel", "outflow_lfcc"),
  site_name = c(
    "Rio Grande at Otowi Bridge, near San Ildefonso, NM",
    "Rio Grande Floodway at San Marcial, NM",
    "Rio Grande Conveyance Channel at San Marcial, NM"
  ),
  stringsAsFactors = FALSE
)

fetch_daily_values <- function(site_row) {
  # read_waterdata_daily uses modern USGS monitoring-location IDs.
  monitoring_location_id <- paste0("USGS-", site_row[["site_no"]])

  raw <- read_waterdata_daily(
    monitoring_location_id = monitoring_location_id,
    parameter_code = parameter_code,
    statistic_id = statistic_id,
    time = c(query_start, query_end),
    properties = c("monitoring_location_id", "time", "value", "qualifier"),
    skipGeometry = TRUE
  )

  if (nrow(raw) == 0) {
    return(data.frame(
      site_no = character(),
      date = as.Date(character()),
      discharge_cfs = numeric(),
      qualifiers = character(),
      stringsAsFactors = FALSE
    ))
  }

  daily <- data.frame(
    site_no = sub("^USGS-", "", raw$monitoring_location_id),
    date = as.Date(raw$time),
    discharge_cfs = as.numeric(raw$value),
    qualifiers = if ("qualifier" %in% names(raw)) as.character(raw$qualifier) else NA_character_,
    stringsAsFactors = FALSE
  )

  # Attach readable site metadata to every daily row.
  daily$short_name <- site_row[["short_name"]]
  daily$role <- site_row[["role"]]
  daily$site_name <- site_row[["site_name"]]
  daily
}

# Download and combine the daily records for all three gages.
daily_list <- lapply(seq_len(nrow(sites)), function(i) fetch_daily_values(sites[i, ]))
daily <- bind_rows(daily_list) |>
  arrange(site_no, date)

cat("Downloaded", comma(nrow(daily)), "daily rows across", nrow(sites), "gages.\n")
sites


## The Completeness Screen

To ensure consistency in the upstream and downstream calculations, we develop a completeness screen: how many days in each calendar year have data from all three gages?



In [ ]:
# Summarize the date span and non-missing daily values for each gage.
site_summary <- daily |>
  group_by(site_no, short_name, role, site_name) |>
  summarize(
    first_date = min(date, na.rm = TRUE),
    last_date = max(date, na.rm = TRUE),
    non_missing_days = sum(!is.na(discharge_cfs)),
    total_rows = n(),
    .groups = "drop"
  ) |>
  mutate(
    span_days = as.integer(last_date - first_date) + 1,
    pct_complete_within_span = round(100 * non_missing_days / span_days, 2)
  )

# Put the three gages side by side so daily joint completeness can be tested.
wide_daily <- daily |>
  select(date, short_name, discharge_cfs) |>
  pivot_wider(names_from = short_name, values_from = discharge_cfs)

expected_days <- data.frame(
  date = seq(min(daily$date), max(daily$date), by = "day")
) |>
  mutate(year = as.integer(format(date, "%Y"))) |>
  count(year, name = "days_in_year")

annual_joint_completeness <- wide_daily |>
  mutate(
    year = as.integer(format(date, "%Y")),
    all_three_present = if_all(all_of(sites$short_name), ~ !is.na(.x))
  ) |>
  group_by(year) |>
  summarize(days_all_three_present = sum(all_three_present), .groups = "drop") |>
  left_join(expected_days, by = "year") |>
  mutate(pct_all_three_present = round(100 * days_all_three_present / days_in_year, 1))

full_years <- annual_joint_completeness |>
  filter(days_in_year %in% c(365, 366)) |>
  pull(year)

qualifying_years <- annual_joint_completeness |>
  filter(
    pct_all_three_present > completeness_threshold,
    year %in% full_years,
    year <= final_analysis_year
  ) |>
  arrange(year) |>
  pull(year)

cat(
  "Qualifying years above", completeness_threshold, "% joint completeness:",
  min(qualifying_years), "-", max(qualifying_years),
  "(", length(qualifying_years), "years )\n"
)

site_summary
annual_joint_completeness |>
  filter(year >= min(qualifying_years) - 2, year <= final_analysis_year) |>
  select(year, days_all_three_present, days_in_year, pct_all_three_present) |>
  head(8)
annual_joint_completeness |>
  filter(year >= min(qualifying_years) - 2, year <= final_analysis_year) |>
  select(year, days_all_three_present, days_in_year, pct_all_three_present) |>
  tail(8)


## Conversion to Annual Water Volume

As mentioned above, the USGS returns data expressed in terms of a rate of flow - average daily mean flow in cubic feet per second. This is a standard reporting measurement, but for our purposes we are interested in total volume. We convert cubic feet per second to acre-feet per day, and then sum the total acre-feet across each entire year for which data is complete.


In [ ]:
# Convert daily mean discharge to daily acre-feet, then aggregate by gage and year.
annual_by_site <- daily |>
  mutate(
    year = as.integer(format(date, "%Y")),
    daily_af = discharge_cfs * acre_feet_per_cfs_day
  ) |>
  group_by(year, short_name) |>
  summarize(
    annual_af = if (all(is.na(daily_af))) NA_real_ else sum(daily_af, na.rm = TRUE),
    .groups = "drop"
  )

annual_wide <- annual_by_site |>
  pivot_wider(names_from = short_name, values_from = annual_af) |>
  rename(
    otowi_af = Otowi,
    san_marcial_floodway_af = `San Marcial Floodway`,
    san_marcial_conveyance_af = `San Marcial Conveyance`
  )

# Keep a continuous public year index, including incomplete recent years.
annual_output <- data.frame(year = seq(min(qualifying_years), final_analysis_year)) |>
  left_join(
    annual_joint_completeness |> select(year, pct_all_three_present),
    by = "year"
  ) |>
  left_join(
    annual_wide |> select(year, otowi_af, san_marcial_floodway_af, san_marcial_conveyance_af),
    by = "year"
  )

flow_columns <- c("otowi_af", "san_marcial_floodway_af", "san_marcial_conveyance_af")

# Mask years that fail the joint-completeness screen before calculating NFR.
annual_output <- annual_output |>
  mutate(across(all_of(flow_columns), ~ if_else(year %in% qualifying_years, .x, NA_real_))) |>
  mutate(
    san_marcial_total_af = san_marcial_floodway_af + san_marcial_conveyance_af,
    net_flow_reduction_af = otowi_af - san_marcial_total_af
  ) |>
  mutate(
    across(
      c(otowi_af, san_marcial_floodway_af, san_marcial_conveyance_af, san_marcial_total_af, net_flow_reduction_af),
      ~ round(.x, 1)
    )
  )

# This CSV is the handoff to the later public notebooks.
write_csv(annual_output, output_path)

cat("Saved annual flow table to", file.path("data", "nfr", basename(output_path)), "\n")
head(annual_output)
tail(annual_output)


## The Results

The figure below shows how much water flows in, how much flows out (the sum of the two San Marcial gages), and the resulting Net Flow Reduction (NFR) data series that results from subtracting outflow from inflow. (Gaps represent years for which data are incomplete.)



In [ ]:
# Reshape the annual table for plotting the two flow series together.
flow_plot <- annual_output |>
  select(year, otowi_af, san_marcial_total_af) |>
  pivot_longer(-year, names_to = "series", values_to = "acre_feet") |>
  mutate(
    series = recode(
      series,
      otowi_af = "Otowi inflow",
      san_marcial_total_af = "San Marcial total outflow"
    )
  )

flow_chart <- ggplot(flow_plot, aes(x = year, y = acre_feet, color = series)) +
  geom_line(linewidth = 0.9) +
  scale_y_continuous(labels = comma) +
  labs(
    title = "Annual inflow and outflow used in the NFR accounting table",
    x = NULL,
    y = "Annual flow (acre-feet)",
    color = NULL
  ) +
  theme_minimal(base_size = 12) +
  theme(legend.position = "bottom")

nfr_chart <- ggplot(annual_output, aes(x = year, y = net_flow_reduction_af)) +
  geom_hline(yintercept = 0, color = "gray50") +
  geom_line(linewidth = 0.9, color = "black") +
  scale_y_continuous(labels = comma) +
  labs(
    title = "Net Flow Reduction = Otowi inflow - combined San Marcial outflow",
    x = "Year",
    y = "NFR (acre-feet)"
  ) +
  theme_minimal(base_size = 12)

print(flow_chart)
print(nfr_chart)


## Conclusion

This notebook creates the base level data on which the rest of the analysis builds. It does not, by itself, tell us anything about how the water is being consumed, but it does allow us to begin disentangling the complex set of factors that must be included in any analysis of how much water the Middle Rio Grande Valley (MRGV) is using, which is foundational to understanding how communities must respond to growing scarcity.
